# Training Classification Models

In this notebook, we explore several approaches using Machine Learning, Deep Learning, and even BERT models to try to classify competencies based on course descriptions.

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.multioutput import ClassifierChain
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score, hamming_loss
import random
import sys

# --- 1. Seed Configuration ---
def set_seed(seed_value=42):
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    random.seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# --- 2. MLP Wrapper to Look Like Scikit-Learn ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, inputs, targets):
        BCE_loss = nn.BCEWithLogitsLoss(reduction='none')(inputs, targets)
        pt = torch.exp(-BCE_loss)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        F_loss = alpha_t * (1 - pt) ** self.gamma * BCE_loss
        return torch.mean(F_loss) if self.reduction == 'mean' else torch.sum(F_loss)

class PyTorchMLPWrapper:
    def __init__(self, input_dim, num_labels, epochs=50, batch_size=16, lr=1e-4):
        self.input_dim = input_dim
        self.num_labels = num_labels
        self.epochs = epochs
        self.batch_size = batch_size
        self.lr = lr
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        
        # "Slim" architecture with BatchNorm
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(128, num_labels)
        ).to(self.device)

    def fit(self, X, y):
        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        y_tensor = torch.tensor(y, dtype=torch.float32).to(self.device)
        dataset = TensorDataset(X_tensor, y_tensor)
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        
        criterion = FocalLoss()
        optimizer = optim.AdamW(self.model.parameters(), lr=self.lr)
        
        self.model.train()
        for epoch in range(self.epochs):
            for batch_X, batch_y in loader:
                optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = criterion(outputs, batch_y)
                loss.backward()
                optimizer.step()
        return self

    def predict_proba(self, X):
        self.model.eval()
        X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
        loader = DataLoader(TensorDataset(X_tensor), batch_size=self.batch_size, shuffle=False)
        all_probs = []
        with torch.no_grad():
            for batch in loader:
                logits = self.model(batch[0])
                probs = torch.sigmoid(logits).cpu().numpy()
                all_probs.append(probs)
        return np.concatenate(all_probs, axis=0)

# --- 3. Master Evaluation Function ---
def evaluate_and_print(y_test, y_proba, mlb_classes):
    """
    Evaluates the model using full metrics for each Top-K setting.
    """
    print(f"\n{'='*60}\n DETAILED RESULTS BY TOP-K \n{'='*60}")

    total_samples = y_test.shape[0]
    k_values = [1, 3, 5, 7, 10]  # K values to analyze
    
    # Compute everything for each K
    for k in k_values:
        print(f"\n>>> TOP-{k} ANALYSIS (Forcing {k} predictions per course) <<<")
        
        # 1. Build the Top-K prediction matrix
        # Create a zero matrix
        y_pred_k = np.zeros_like(y_test)
        
        # Get indices of the K highest probabilities
        # np.argsort sorts from smallest to largest; we take the last K ([-k:])
        top_k_indices = np.argsort(y_proba, axis=1)[:, -k:]
        
        # Fill with 1 only at Top-K indices for each sample
        for i in range(total_samples):
            y_pred_k[i, top_k_indices[i]] = 1
            
        # 2. Compute classification metrics (F1, Hamming) for this K
        f1_mic = f1_score(y_test, y_pred_k, average='micro')
        f1_mac = f1_score(y_test, y_pred_k, average='macro', zero_division=0)
        h_loss = hamming_loss(y_test, y_pred_k)
        
        # 3. Compute ranking metrics (Precision@K and Partial Hit)
        # (Logic reused for consistency)
        total_hits = 0
        samples_with_hit = 0
        for i in range(total_samples):
            true_indices = np.where(y_test[i] == 1)[0]
            pred_indices = top_k_indices[i]  # These are the Top-K indices
            
            hits = len(set(pred_indices) & set(true_indices))
            total_hits += hits
            if hits > 0:
                samples_with_hit += 1
        
        precision_at_k = total_hits / (total_samples * k)
        hit_rate_at_k = (samples_with_hit / total_samples) * 100
        
        # 4. Print results block
        print(f"{'-'*40}")
        print(f"Hamming Loss:        {h_loss:.4f}")
        print(f"F1 Score (Micro):    {f1_mic:.4f}")
        print(f"F1 Score (Macro):    {f1_mac:.4f}")
        print(f"{'-'*40}")
        print(f"Precision@{k}:        {precision_at_k:.4f} (Hits within the {k} guesses)")
        print(f"Partial Hit@{k}:      {hit_rate_at_k:.2f}% (Courses with at least 1 hit)")
        print(f"{'-'*40}")
    
# --- 4. Main Function (Simplified API) ---
def run_experiment(model_name, X_train, y_train, X_test, y_test, mlb_classes):
    print(f"\n{'='*40}\nRunning: {model_name}\n{'='*40}")
    
    # 1. Select model
    if model_name == 'xgb':
        base = XGBClassifier(n_estimators=250, max_depth=5, random_state=42, eval_metric='logloss')
        model = ClassifierChain(base, order='random', random_state=42)
    elif model_name == 'rf':
        base = RandomForestClassifier(n_estimators=250, random_state=42)
        model = ClassifierChain(base, order='random', random_state=42)
    elif model_name == 'gb':
        base = GradientBoostingClassifier(n_estimators=250, random_state=42)
        model = ClassifierChain(base, order='random', random_state=42)
    elif model_name == 'mlp':
        # Our wrapper handles everything
        model = PyTorchMLPWrapper(input_dim=X_train.shape[1], num_labels=y_train.shape[1])
    else:
        raise ValueError("Unknown model. Use: 'xgb', 'rf', 'gb', or 'mlp'")

    # 2. Train
    print("Training...")
    model.fit(X_train, y_train)
    
    # 3. Predict
    print("Generating probabilities...")
    y_proba = model.predict_proba(X_test)
    
    # 4. Evaluate
    print("Evaluating...")
    evaluate_and_print(y_test, y_proba, mlb_classes)


In [3]:
import random
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

# ML and NLP imports
import torch
from gensim.models import Word2Vec
from transformers import BertTokenizer, BertModel
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split

# --- Initial Configuration ---
# Download NLTK resources silently
try:
    nltk.data.find('tokenizers/punkt')
    nltk.data.find('corpus/stopwords')
    nltk.data.find('punkt_tab')
except LookupError:
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt_tab', quiet=True)

# Set seeds
def set_seed(seed_value=42):
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    random.seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

# --- Constants ---
STOP_WORDS_PT = set(stopwords.words('portuguese'))
DOMAIN_STOP_WORDS = {
    'curso', 'aprendizagem', 'educação', 'gestão', 'avaliação', 'pessoas',
    'científica', 'inclusão', 'trabalho', 'ensino', 'servidores', 'uso',
    'objetivo', 'conhecimento', 'público', 'formação', 'conceitos'
}
ALL_STOP_WORDS = list(STOP_WORDS_PT.union(DOMAIN_STOP_WORDS))

# --- Helper Functions ---

def _load_and_filter_data(csv_path, min_course_count=5):
    """Loads, groups, and filters the data."""
    print("-> Loading and filtering data...")
    try:
        df = pd.read_csv(csv_path)
    except FileNotFoundError:
        sys.exit("Error: CSV file not found.")
        
    df = df.dropna(subset=['courseName', 'comp_name', 'courseDescription'])
    
    # Group by course
    df_agg = df.groupby('courseName', as_index=False).agg({
        'courseDescription': 'first',
        'comp_name': lambda x: list(set(x))
    })
    
    # Filter rare competencies
    counts = df_agg['comp_name'].explode().value_counts()
    rare_comps = counts[counts < min_course_count].index
    
    df_agg['comp_name_filtered'] = df_agg['comp_name'].apply(
        lambda x: [c for c in x if c not in rare_comps]
    )
    # Remove courses that became empty after filtering
    df_filtered = df_agg[df_agg['comp_name_filtered'].apply(len) > 0].copy()
    
    # Build combined text
    df_filtered['combinedText'] = df_filtered['courseDescription'].fillna('')
    
    print(f"Processed data: {len(df_filtered)} courses remaining.")
    return df_filtered

def _get_tfidf_features(texts):
    """Generates TF-IDF features."""
    print("-> Generating TF-IDF features...")
    tfidf = TfidfVectorizer(
        max_features=2000, ngram_range=(1, 2), min_df=5, max_df=0.7,
        stop_words=ALL_STOP_WORDS, sublinear_tf=True
    )
    # Return dense array for PyTorch compatibility
    return tfidf.fit_transform(texts).toarray()

def _get_word2vec_features(texts, vector_size=300):
    """Generates Word2Vec features (mean pooling)."""
    print(f"-> Generating Word2Vec features (dim={vector_size})...")
    
    def clean_text(text):
        text = text.lower().translate(str.maketrans('', '', string.punctuation))
        return [w for w in word_tokenize(text) if w not in STOP_WORDS_PT and w.isalpha()]

    tokens = [clean_text(t) for t in texts]
    
    model = Word2Vec(
        sentences=tokens,
        vector_size=vector_size,
        window=5,
        min_count=2,
        workers=4
    )
    
    embeddings = []
    for t in tokens:
        valid = [model.wv[w] for w in t if w in model.wv]
        embeddings.append(np.mean(valid, axis=0) if valid else np.zeros(vector_size))
        
    return np.vstack(embeddings)

def _get_bert_embeddings(texts):
    """Generates BERT [CLS] embeddings."""
    print("-> Generating BERT features (this may take some time)...")
    model_name = 'neuralmind/bert-base-portuguese-cased'
    tokenizer = BertTokenizer.from_pretrained(model_name)
    model = BertModel.from_pretrained(model_name)
    
    # Process in batches to save memory
    batch_size = 32
    all_embeddings = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        inputs = tokenizer(
            batch_texts,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=128
        )
        with torch.no_grad():
            outputs = model(**inputs)
        all_embeddings.append(outputs.last_hidden_state[:, 0, :].numpy())
        
    return np.vstack(all_embeddings)

# --- PIPELINE ---

def get_data_pipeline(embedding_type='tfidf', csv_path="dataset_ifrn_artigo.csv"):
    """
    Full data preparation pipeline.
    Args:
        embedding_type: 'tfidf', 'word2vec', or 'bert'
    Returns:
        X_train, X_test, y_train, y_test, mlb_classes (class names)
    """
    # 1. Load and filter
    df = _load_and_filter_data(csv_path, min_course_count=5)
    texts = df['combinedText'].tolist()
    
    # 2. Build features (X)
    if embedding_type == 'tfidf':
        X = _get_tfidf_features(texts)
    elif embedding_type == 'word2vec':
        X = _get_word2vec_features(texts, vector_size=300)
    elif embedding_type == 'bert':
        X = _get_bert_embeddings(texts)
    else:
        raise ValueError("Embedding must be one of: 'tfidf', 'word2vec', or 'bert'")
    
    # 3. Build labels (y)
    print("-> Binarizing labels...")
    mlb = MultiLabelBinarizer()
    y = mlb.fit_transform(df['comp_name_filtered'])
    
    # 4. Split
    print("-> Splitting train and test sets...")
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    
    print(f"Pipeline completed. X_train shape: {X_train.shape}")
    return X_train, X_test, y_train, y_test, mlb.classes_


## Machine Learning com TF-IDF Vectorizer

In [4]:
#1. Obtain data prepared for TF-IDF
X_train, X_test, y_train, y_test, classes = get_data_pipeline('tfidf')

-> Loading and filtering data...
Processed data: 337 courses remaining.
-> Generating TF-IDF features...
-> Binarizing labels...
-> Splitting train and test sets...
Pipeline completed. X_train shape: (269, 817)


In [5]:
# 2.1 Run Gradient Boosting - TF-IDF
run_experiment('gb', X_train, y_train, X_test, y_test, classes)


Running: gb
Training...
Generating probabilities...
Evaluating...

 DETAILED RESULTS BY TOP-K 

>>> TOP-1 ANALYSIS (Forcing 1 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0622
F1 Score (Micro):    0.1515
F1 Score (Macro):    0.0625
----------------------------------------
Precision@1:        0.2941 (Hits within the 1 guesses)
Partial Hit@1:      29.41% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-3 ANALYSIS (Forcing 3 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0827
F1 Score (Micro):    0.2550
F1 Score (Macro):    0.1063
----------------------------------------
Precision@3:        0.2500 (Hits within the 3 guesses)
Partial Hit@3:      55.88% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-5 ANALYSIS (Forcing 5 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.1138
F1 Score (Micro):   

In [6]:
# 2.2 Run Random Forest - TF-IDF
run_experiment('rf', X_train, y_train, X_test, y_test, classes)


Running: rf
Training...
Generating probabilities...
Evaluating...

 DETAILED RESULTS BY TOP-K 

>>> TOP-1 ANALYSIS (Forcing 1 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0583
F1 Score (Micro):    0.2045
F1 Score (Macro):    0.0448
----------------------------------------
Precision@1:        0.3971 (Hits within the 1 guesses)
Partial Hit@1:      39.71% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-3 ANALYSIS (Forcing 3 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0782
F1 Score (Micro):    0.2950
F1 Score (Macro):    0.1057
----------------------------------------
Precision@3:        0.2892 (Hits within the 3 guesses)
Partial Hit@3:      61.76% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-5 ANALYSIS (Forcing 5 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.1038
F1 Score (Micro):   

In [7]:
# 2.3 Run XGBoost - TF-IDF
run_experiment('xgb', X_train, y_train, X_test, y_test, classes)


Running: xgb
Training...
Generating probabilities...
Evaluating...

 DETAILED RESULTS BY TOP-K 

>>> TOP-1 ANALYSIS (Forcing 1 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0610
F1 Score (Micro):    0.1667
F1 Score (Macro):    0.0568
----------------------------------------
Precision@1:        0.3235 (Hits within the 1 guesses)
Partial Hit@1:      32.35% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-3 ANALYSIS (Forcing 3 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0799
F1 Score (Micro):    0.2800
F1 Score (Macro):    0.1350
----------------------------------------
Precision@3:        0.2745 (Hits within the 3 guesses)
Partial Hit@3:      58.82% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-5 ANALYSIS (Forcing 5 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.1127
F1 Score (Micro):  

In [8]:
# 2.4 Run MLP - TF-IDF
run_experiment('mlp', X_train, y_train, X_test, y_test, classes)


Running: mlp
Training...
Generating probabilities...
Evaluating...

 DETAILED RESULTS BY TOP-K 

>>> TOP-1 ANALYSIS (Forcing 1 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0622
F1 Score (Micro):    0.1515
F1 Score (Macro):    0.0229
----------------------------------------
Precision@1:        0.2941 (Hits within the 1 guesses)
Partial Hit@1:      29.41% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-3 ANALYSIS (Forcing 3 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0821
F1 Score (Micro):    0.2600
F1 Score (Macro):    0.0993
----------------------------------------
Precision@3:        0.2549 (Hits within the 3 guesses)
Partial Hit@3:      54.41% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-5 ANALYSIS (Forcing 5 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.1115
F1 Score (Micro):  

## Machine Learning com Word2Vec

In [9]:
# 1. Get data prepared for Word2Vec
X_train, X_test, y_train, y_test, classes = get_data_pipeline('word2vec')

-> Loading and filtering data...
Processed data: 337 courses remaining.
-> Generating Word2Vec features (dim=300)...
-> Binarizing labels...
-> Splitting train and test sets...
Pipeline completed. X_train shape: (269, 300)


In [10]:
# 2.1 Run Gradient Boosting - Word2Vec
run_experiment('gb', X_train, y_train, X_test, y_test, classes)


Running: gb
Training...
Generating probabilities...
Evaluating...

 DETAILED RESULTS BY TOP-K 

>>> TOP-1 ANALYSIS (Forcing 1 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0633
F1 Score (Micro):    0.1364
F1 Score (Macro):    0.0271
----------------------------------------
Precision@1:        0.2647 (Hits within the 1 guesses)
Partial Hit@1:      26.47% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-3 ANALYSIS (Forcing 3 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0871
F1 Score (Micro):    0.2150
F1 Score (Macro):    0.0537
----------------------------------------
Precision@3:        0.2108 (Hits within the 3 guesses)
Partial Hit@3:      48.53% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-5 ANALYSIS (Forcing 5 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.1188
F1 Score (Micro):   

In [11]:
# 2.2 Run Random Forest - Word2Vec
run_experiment('rf', X_train, y_train, X_test, y_test, classes)


Running: rf
Training...
Generating probabilities...
Evaluating...

 DETAILED RESULTS BY TOP-K 

>>> TOP-1 ANALYSIS (Forcing 1 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0605
F1 Score (Micro):    0.1742
F1 Score (Macro):    0.0259
----------------------------------------
Precision@1:        0.3382 (Hits within the 1 guesses)
Partial Hit@1:      33.82% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-3 ANALYSIS (Forcing 3 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0832
F1 Score (Micro):    0.2500
F1 Score (Macro):    0.0735
----------------------------------------
Precision@3:        0.2451 (Hits within the 3 guesses)
Partial Hit@3:      52.94% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-5 ANALYSIS (Forcing 5 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.1143
F1 Score (Micro):   

In [12]:
# 2.3 Run XGBoost - Word2Vec
run_experiment('xgb', X_train, y_train, X_test, y_test, classes)


Running: xgb
Training...
Generating probabilities...
Evaluating...

 DETAILED RESULTS BY TOP-K 

>>> TOP-1 ANALYSIS (Forcing 1 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0594
F1 Score (Micro):    0.1894
F1 Score (Macro):    0.0642
----------------------------------------
Precision@1:        0.3676 (Hits within the 1 guesses)
Partial Hit@1:      36.76% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-3 ANALYSIS (Forcing 3 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0855
F1 Score (Micro):    0.2300
F1 Score (Macro):    0.0876
----------------------------------------
Precision@3:        0.2255 (Hits within the 3 guesses)
Partial Hit@3:      50.00% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-5 ANALYSIS (Forcing 5 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.1149
F1 Score (Micro):  

In [13]:
# 2.4 Run MLP - Word2Vec
run_experiment('mlp', X_train, y_train, X_test, y_test, classes)


Running: mlp
Training...
Generating probabilities...
Evaluating...

 DETAILED RESULTS BY TOP-K 

>>> TOP-1 ANALYSIS (Forcing 1 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0627
F1 Score (Micro):    0.1439
F1 Score (Macro):    0.0084
----------------------------------------
Precision@1:        0.2794 (Hits within the 1 guesses)
Partial Hit@1:      27.94% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-3 ANALYSIS (Forcing 3 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0882
F1 Score (Micro):    0.2050
F1 Score (Macro):    0.0319
----------------------------------------
Precision@3:        0.2010 (Hits within the 3 guesses)
Partial Hit@3:      41.18% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-5 ANALYSIS (Forcing 5 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.1182
F1 Score (Micro):  

# Bert

In [14]:
#1. Get data prepared for BERT embeddings
X_train, X_test, y_train, y_test, classes = get_data_pipeline('bert')

-> Loading and filtering data...
Processed data: 337 courses remaining.
-> Generating BERT features (this may take some time)...
-> Binarizing labels...
-> Splitting train and test sets...
Pipeline completed. X_train shape: (269, 768)


In [15]:
# 2.1 Run Gradient Boosting - BERT
run_experiment('gb', X_train, y_train, X_test, y_test, classes)


Running: gb
Training...
Generating probabilities...
Evaluating...

 DETAILED RESULTS BY TOP-K 

>>> TOP-1 ANALYSIS (Forcing 1 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0605
F1 Score (Micro):    0.1742
F1 Score (Macro):    0.0758
----------------------------------------
Precision@1:        0.3382 (Hits within the 1 guesses)
Partial Hit@1:      33.82% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-3 ANALYSIS (Forcing 3 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0816
F1 Score (Micro):    0.2650
F1 Score (Macro):    0.1209
----------------------------------------
Precision@3:        0.2598 (Hits within the 3 guesses)
Partial Hit@3:      55.88% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-5 ANALYSIS (Forcing 5 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.1115
F1 Score (Micro):   

In [16]:
# 2.2 Run Random Forest - BERT
run_experiment('rf', X_train, y_train, X_test, y_test, classes)


Running: rf
Training...
Generating probabilities...
Evaluating...

 DETAILED RESULTS BY TOP-K 

>>> TOP-1 ANALYSIS (Forcing 1 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0549
F1 Score (Micro):    0.2500
F1 Score (Macro):    0.0543
----------------------------------------
Precision@1:        0.4853 (Hits within the 1 guesses)
Partial Hit@1:      48.53% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-3 ANALYSIS (Forcing 3 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0727
F1 Score (Micro):    0.3450
F1 Score (Macro):    0.1247
----------------------------------------
Precision@3:        0.3382 (Hits within the 3 guesses)
Partial Hit@3:      66.18% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-5 ANALYSIS (Forcing 5 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0960
F1 Score (Micro):   

In [17]:
# 2.3 Run XGBoost - BERT
run_experiment('xgb', X_train, y_train, X_test, y_test, classes)


Running: xgb
Training...
Generating probabilities...
Evaluating...

 DETAILED RESULTS BY TOP-K 

>>> TOP-1 ANALYSIS (Forcing 1 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0560
F1 Score (Micro):    0.2348
F1 Score (Macro):    0.0906
----------------------------------------
Precision@1:        0.4559 (Hits within the 1 guesses)
Partial Hit@1:      45.59% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-3 ANALYSIS (Forcing 3 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0738
F1 Score (Micro):    0.3350
F1 Score (Macro):    0.1693
----------------------------------------
Precision@3:        0.3284 (Hits within the 3 guesses)
Partial Hit@3:      66.18% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-5 ANALYSIS (Forcing 5 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.1021
F1 Score (Micro):  

In [18]:
# 2.4 Run MLP - MLP
run_experiment('mlp', X_train, y_train, X_test, y_test, classes)


Running: mlp
Training...
Generating probabilities...
Evaluating...

 DETAILED RESULTS BY TOP-K 

>>> TOP-1 ANALYSIS (Forcing 1 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0527
F1 Score (Micro):    0.2803
F1 Score (Macro):    0.1116
----------------------------------------
Precision@1:        0.5441 (Hits within the 1 guesses)
Partial Hit@1:      54.41% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-3 ANALYSIS (Forcing 3 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.0744
F1 Score (Micro):    0.3300
F1 Score (Macro):    0.1413
----------------------------------------
Precision@3:        0.3235 (Hits within the 3 guesses)
Partial Hit@3:      64.71% (Courses with at least 1 hit)
----------------------------------------

>>> TOP-5 ANALYSIS (Forcing 5 predictions per course) <<<
----------------------------------------
Hamming Loss:        0.1060
F1 Score (Micro):  

### Partial Hit Results by Top-K

Below we present the percentage of courses that had at least one correctly predicted competency within the *K* highest probabilities indicated by the model.

#### Table 1: Precision @ Top-1 (Accuracy of the 1st choice)

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | 39.71% | 30.88% | 48.53% |
| **XGBoost** | 32.35% | 36.76% | 45.59% |
| **MLP (Deep Learning)** | 29.41% | 29.41% | **50.00%** |
| **Gradient Boosting** | 29.41% | 30.88% | 33.82% |

#### Table 2: Accuracy @ Top-3

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | 61.76% | 47.06% | **66.18%** |
| **XGBoost** | 58.82% | 54.41% | **66.18%** |
| **MLP (Deep Learning)** | 54.41% | 38.24% | 61.76% |

| **Gradient Boosting** | 55.88% | 52.94% | 55.88% |

#### Table 3: Accuracy @ Top-5

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | 73.53% | 60.29% | **80.88%** |
| **XGBoost** | 60.29% | 64.71% | 73.53% |
| **MLP (Deep Learning)** | 64.71% | 52.94% | 73.53% |
| **Gradient Boosting** | 64.71% | 61.76% | 67.65% |

#### Table 4: Accuracy @ Top-7

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | 79.41% | 72.06% | **83.82%** |
| **XGBoost** | 69.12% | 66.18% | 80.88% |
| **MLP (Deep Learning)** | 69.12% | 60.29% | 76.47% |
| **Gradient Boosting** | 75.00% | 66.18% | 73.53% |

#### Table 5: Accuracy @ Top-10

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | **86.76%** | 72.06% | **86.76%** |
| **XGBoost** | 72.06% | 75.00% | **86.76%** |
| **MLP (Deep Learning)** | 76.47% | 67.65% | 82.35% |
| **Gradient Boosting** | 80.88% | 76.47% | 79.41% |

## Precision Results @ Top-K

The Precision@K metric indicates the average proportion of correctly predicted competencies among the *K* competencies suggested by the model for each course.

### Table 1 — Precision @ Top-1

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | **0.3971** | 0.3088 | **0.4853** |
| **XGBoost** | 0.3235 | 0.3676 | 0.4559 |
| **MLP (Deep Learning)** | 0.2941 | 0.2941 | 0.4559 |
| **Gradient Boosting** | 0.2941 | 0.3088 | 0.3382 |

### Table 2 — Accuracy @ Top-3

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | **0.2892** | 0.2304 | **0.3382** |
| **XGBoost** | 0.2745 | **0.2745** | 0.3284 |
| **MLP (Deep Learning)** | 0.2549 | 0.1863 | **0.3480** |
| **Gradient Boosting** | 0.2500 | 0.2353 | 0.2598 |

### Table 3 — Accuracy @ Top-5

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | **0.2382** | 0.1765 | **0.2794** |
| **XGBoost** | 0.1912 | **0.2029** | 0.2471 |
| **MLP (Deep Learning)** | 0.1971 | 0.1559 | 0.2618 |
| **Gradient Boosting** | 0.1853 | 0.1765 | 0.1971 |

### Table 4 — Accuracy @ Top-7

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | **0.1954** | **0.1576** | **0.2206** |
| **XGBoost** | 0.1534 | 0.1576 | 0.2059 |
| **MLP (Deep Learning)** | 0.1597 | 0.1387 | 0.2059 |
| **Gradient Boosting** | 0.1660 | 0.1366 | 0.1618 |

### Table 5 — Accuracy @ Top-10

| Model | TF-IDF | Word2Vec | BERT |
| :--- | :---: | :---: | :---: |
| **Random Forest** | **0.1603** | 0.1368 | **0.1750** | | **XGBoost** | 0.1221 | 0.1235 | 0.1662 |
| **MLP (Deep Learning)** | 0.1353 | 0.1206 | 0.1603 |
| **Gradient Boosting** | 0.1338 | 0.1221 | 0.1338 |

> Unlike the hit rate metric, Precision@K explicitly penalizes the inclusion of incorrect skills, offering a more rigorous assessment of the quality of the lists generated by the models.